### Task 1: Load and Preprocess Texts

First, we'll import the necessary libraries, define the URLs for the books, and create a function `load_texts()` to fetch the content, clean it using regular expressions, and extract the relevant parts between 'START' and 'END' markers.

In [ ]:
import requests
import re

# Define the URLs for the Lewis Carroll books
book_urls = [
    'https://www.gutenberg.org/cache/epub/11/pg11.txt', # Alice’s Adventures in Wonderland
    'https://www.gutenberg.org/cache/epub/12/pg12.txt', # THROUGH THE LOOKING-GLASS And What Alice Found There
    'https://www.gutenberg.org/cache/epub/29042/pg29042.txt' # A Tangled Tale
]


In [ ]:
import requests
import re

def load_texts(urls):
    corpus = []
    for url in urls:
        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
            raw_text = response.text

            # Step 1: Extract content between Project Gutenberg boilerplate
            # Using more robust and non-greedy regex for start/end markers, with DOTALL for multiline
            gutenberg_start_match = re.search(r'\*{3}\s*START OF THE PROJECT GUTENBERG EBOOK.*?\*{3}', raw_text, re.IGNORECASE | re.DOTALL)
            gutenberg_end_match = re.search(r'\*{3}\s*END OF THE PROJECT GUTENBERG EBOOK.*?\*{3}', raw_text, re.IGNORECASE | re.DOTALL)

            gutenberg_content = raw_text
            if gutenberg_start_match and gutenberg_end_match and gutenberg_start_match.end() < gutenberg_end_match.start():
                gutenberg_content = raw_text[gutenberg_start_match.end() : gutenberg_end_match.start()].strip()
            elif gutenberg_start_match:
                gutenberg_content = raw_text[gutenberg_start_match.end():].strip()
            elif gutenberg_end_match:
                gutenberg_content = raw_text[:gutenberg_end_match.start()].strip()
            # Else gutenberg_content remains raw_text

            # Step 2: Further refine by finding the actual start and end of the narrative content
            # Based on the user's hint 'START' and 'END' referring to actual narrative.
            story_start_offset = 0
            story_end_offset = len(gutenberg_content) # Use original length for slicing

            # Book identification based on unique phrases in raw_text
            if "Alice's Adventures in Wonderland" in raw_text: # Book 1
                # Look for 'CHAPTER I. Down the Rabbit-Hole' as the true narrative start
                match_start = re.search(r'CHAPTER I\.\s*Down the Rabbit-Hole', gutenberg_content, re.IGNORECASE | re.DOTALL)
                if match_start:
                    story_start_offset = match_start.start()

                # Look for 'THE END.' as the narrative end
                match_end = re.search(r'THE END\.', gutenberg_content[story_start_offset:], re.IGNORECASE | re.DOTALL)
                if match_end:
                    story_end_offset = story_start_offset + match_end.end()

            elif "Through the Looking-Glass" in raw_text: # Book 2
                # Corrected regex for 'CHAPTER I. LOOKING-GLASS HOUSE' (no period after House)
                match_start = re.search(r'CHAPTER I\.\s*LOOKING-GLASS HOUSE', gutenberg_content, re.IGNORECASE | re.DOTALL)
                if match_start:
                    story_start_offset = match_start.start()

                # Look for 'THE END.' as the narrative end
                match_end = re.search(r'THE END\.', gutenberg_content[story_start_offset:], re.IGNORECASE | re.DOTALL)
                if match_end:
                    story_end_offset = story_start_offset + match_end.end()

            elif "A Tangled Tale" in raw_text: # Book 3
                # Look for 'KNOT I. The Dedication' or simply 'KNOT I.' as the narrative start
                match_start = re.search(r'(KNOT I\.\s*The Dedication|KNOT I\.)', gutenberg_content, re.IGNORECASE | re.DOTALL)
                if match_start:
                    story_start_offset = match_start.start()

                # A Tangled Tale often has 'ANSWERS TO KNOTS' after the story, so end before that.
                answers_match = re.search(r'ANSWERS TO KNOTS', gutenberg_content[story_start_offset:], re.IGNORECASE | re.DOTALL)
                if answers_match:
                    story_end_offset = story_start_offset + answers_match.start()
                else:
                    # Fallback to 'THE END.' if Answers section is not found
                    match_end = re.search(r'THE END\.', gutenberg_content[story_start_offset:], re.IGNORECASE | re.DOTALL)
                    if match_end:
                        story_end_offset = story_start_offset + match_end.end()

            relevant_text = gutenberg_content[story_start_offset:story_end_offset].strip()

            # Step 3: Clean non-words using regular expressions from the refined relevant_text
            # Keep letters, numbers, and basic punctuation (.,?!), remove extra spaces
            cleaned_text = re.sub(r'[^a-zA-Z0-9.,?!\s]', '', relevant_text)
            cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
            corpus.append(cleaned_text)

        except requests.exceptions.RequestException as e:
            print(f"Error fetching {url}: {e}")
            corpus.append("") # Append empty string for failed loads
        except Exception as e:
            print(f"An unexpected error occurred for {url}: {e}")
            corpus.append("")
    return corpus

# Load and clean the texts
books_corpus = load_texts(book_urls)

### Task 2: Print the first 200 characters of each text

Now, let's display the beginning of each processed book to verify the loading and initial cleaning.

In [ ]:
for i, text in enumerate(books_corpus):
    print(f"--- Book {i+1} (First 200 characters) ---")
    print(text[:200])
    print("\n")

--- Book 1 (First 200 characters) ---
CHAPTER I. Down the RabbitHole CHAPTER II. The Pool of Tears CHAPTER III. A CaucusRace and a Long Tale CHAPTER IV. The Rabbit Sends in a Little Bill CHAPTER V. Advice from a Caterpillar CHAPTER VI. Pi


--- Book 2 (First 200 characters) ---
CHAPTER I. LookingGlass house CHAPTER II. The Garden of Live Flowers CHAPTER III. LookingGlass Insects CHAPTER IV. Tweedledum And Tweedledee CHAPTER V. Wool and Water CHAPTER VI. Humpty Dumpty CHAPTER


--- Book 3 (First 200 characters) ---
KNOT I. EXCELSIOR. Goblin, lead them up and down. The ruddy glow of sunset was already fading into the sombre shadows of night, when two travellers might have been observed swiftlyat a pace of six mil




### Task 3: Tokenize the text and print the first 150 tokens of each book

To tokenize the text, we'll use NLTK's `word_tokenize` function. We need to download the 'punkt' tokenizer models first if they aren't already available.

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

# Download the 'punkt' tokenizer models if not already downloaded
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# Explicitly download 'punkt_tab' as it's sometimes needed by word_tokenize
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

tokenized_books = []
for i, text in enumerate(books_corpus):
    # Convert text to lowercase before tokenization for consistency
    tokens = word_tokenize(text.lower())
    tokenized_books.append(tokens)

    print(f"--- Book {i+1} (First 150 tokens) ---")
    print(tokens[:150])
    print("\n")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


--- Book 1 (First 150 tokens) ---
['chapter', 'i.', 'down', 'the', 'rabbithole', 'chapter', 'ii', '.', 'the', 'pool', 'of', 'tears', 'chapter', 'iii', '.', 'a', 'caucusrace', 'and', 'a', 'long', 'tale', 'chapter', 'iv', '.', 'the', 'rabbit', 'sends', 'in', 'a', 'little', 'bill', 'chapter', 'v.', 'advice', 'from', 'a', 'caterpillar', 'chapter', 'vi', '.', 'pig', 'and', 'pepper', 'chapter', 'vii', '.', 'a', 'mad', 'teaparty', 'chapter', 'viii', '.', 'the', 'queens', 'croquetground', 'chapter', 'ix', '.', 'the', 'mock', 'turtles', 'story', 'chapter', 'x.', 'the', 'lobster', 'quadrille', 'chapter', 'xi', '.', 'who', 'stole', 'the', 'tarts', '?', 'chapter', 'xii', '.', 'alices', 'evidence', 'chapter', 'i.', 'down', 'the', 'rabbithole', 'alice', 'was', 'beginning', 'to', 'get', 'very', 'tired', 'of', 'sitting', 'by', 'her', 'sister', 'on', 'the', 'bank', ',', 'and', 'of', 'having', 'nothing', 'to', 'do', 'once', 'or', 'twice', 'she', 'had', 'peeped', 'into', 'the', 'book', 'her', 'sister', '

### Task 4: Remove Stopwords using NLTK

We will now remove common English stopwords from our tokenized texts. We'll use NLTK's `stopwords` corpus and then verify the removal of some example words.

In [ ]:
import nltk
from nltk.corpus import stopwords

# Download NLTK stopwords if not already downloaded
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

english_stopwords = set(stopwords.words('english'))

cleaned_tokenized_books = []

# Ensure tokenized_books is populated before proceeding
if not tokenized_books:
    print("Tokenization failed in the previous step. Cannot perform stopword removal.")
else:
    for i, tokens in enumerate(tokenized_books):
        filtered_tokens = [word for word in tokens if word not in english_stopwords]
        cleaned_tokenized_books.append(filtered_tokens)

        print(f"--- Verification for Book {i+1} after stopword removal ---")
        # Check for removal of some example stopwords
        example_stopwords = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'the', 'a', 'an']
        for sw in example_stopwords:
            original_count = tokens.count(sw)
            filtered_count = filtered_tokens.count(sw)
            print(f"Stopword '{sw}': Original count = {original_count}, Filtered count = {filtered_count}")
        print("\n")

    print("Stopword removal complete. The 'cleaned_tokenized_books' list now contains the texts without stopwords.")

--- Verification for Book 1 after stopword removal ---
Stopword 'i': Original count = 387, Filtered count = 0
Stopword 'me': Original count = 64, Filtered count = 0
Stopword 'my': Original count = 56, Filtered count = 0
Stopword 'myself': Original count = 7, Filtered count = 0
Stopword 'we': Original count = 25, Filtered count = 0
Stopword 'our': Original count = 8, Filtered count = 0
Stopword 'ours': Original count = 1, Filtered count = 0
Stopword 'ourselves': Original count = 0, Filtered count = 0
Stopword 'the': Original count = 1532, Filtered count = 0
Stopword 'a': Original count = 614, Filtered count = 0
Stopword 'an': Original count = 51, Filtered count = 0


--- Verification for Book 2 after stopword removal ---
Stopword 'i': Original count = 508, Filtered count = 0
Stopword 'me': Original count = 88, Filtered count = 0
Stopword 'my': Original count = 75, Filtered count = 0
Stopword 'myself': Original count = 7, Filtered count = 0
Stopword 'we': Original count = 36, Filtered co

### Task 5: Using PorterStemmer(), print the first 50 stemmed tokens

Now, we'll apply stemming to our cleaned, tokenized texts using NLTK's `PorterStemmer` to reduce words to their root form.

In [ ]:
import nltk
from nltk.stem import PorterStemmer

porter = PorterStemmer()

stemmed_books = []

for i, tokens in enumerate(cleaned_tokenized_books):
    stemmed_tokens = [porter.stem(word) for word in tokens]
    stemmed_books.append(stemmed_tokens)

    print(f"--- Book {i+1} (First 50 stemmed tokens) ---")
    print(stemmed_tokens[:50])
    print("\n")

--- Book 1 (First 50 stemmed tokens) ---
['chapter', 'i.', 'rabbithol', 'chapter', 'ii', '.', 'pool', 'tear', 'chapter', 'iii', '.', 'caucusrac', 'long', 'tale', 'chapter', 'iv', '.', 'rabbit', 'send', 'littl', 'bill', 'chapter', 'v.', 'advic', 'caterpillar', 'chapter', 'vi', '.', 'pig', 'pepper', 'chapter', 'vii', '.', 'mad', 'teaparti', 'chapter', 'viii', '.', 'queen', 'croquetground', 'chapter', 'ix', '.', 'mock', 'turtl', 'stori', 'chapter', 'x.', 'lobster', 'quadril']


--- Book 2 (First 50 stemmed tokens) ---
['chapter', 'i.', 'lookingglass', 'hous', 'chapter', 'ii', '.', 'garden', 'live', 'flower', 'chapter', 'iii', '.', 'lookingglass', 'insect', 'chapter', 'iv', '.', 'tweedledum', 'tweedlede', 'chapter', 'v.', 'wool', 'water', 'chapter', 'vi', '.', 'humpti', 'dumpti', 'chapter', 'vii', '.', 'lion', 'unicorn', 'chapter', 'viii', '.', 'invent', 'chapter', 'ix', '.', 'queen', 'alic', 'chapter', 'x.', 'shake', 'chapter', 'xi', '.', 'wake']


--- Book 3 (First 50 stemmed tokens) ---

### Task 6: Lemmatization using spaCy

We'll use spaCy's pre-trained English model to perform lemmatization. First, we need to ensure spaCy and its model are installed and loaded.

In [ ]:
import spacy
import subprocess
import sys

# Function to install packages if not already installed
def install_package(package):
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"{package} installed successfully.")

# Install spaCy if not already installed
install_package("spacy")

# Load the English model, download if not present
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading spaCy model 'en_core_web_sm'...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

lemmatized_books = []

for i, text in enumerate(books_corpus):
    # Process the entire cleaned text with spaCy
    doc = nlp(text)
    # Extract lemmas, converting to lowercase and filtering out punctuation/spaces if desired
    # For direct comparison with stemmed words, we should use the cleaned tokens
    # However, spaCy's lemmatizer works best on raw text input for context.
    # Let's take the approach of processing the already cleaned, non-stopword tokens for lemma extraction

    # Re-tokenize and clean for spaCy processing, or use the original cleaned_text
    # For best lemmatization results, it's often better to pass the raw (but cleaned) string to nlp()
    # and then extract lemmas from the processed Doc object.
    # Let's use the 'cleaned_tokenized_books' (which are already lowercased and stopwords removed)
    # and then process them through spaCy token by token for lemma, maintaining consistency.

    # Alternative: re-process the raw cleaned text, then filter for non-stopwords and get lemmas
    # Since the instruction asks to print the first 50 lemmatized tokens, we will re-process
    # the 'books_corpus' (cleaned but not tokenized/stemmed) to get the most accurate lemmas.
    # Then, we will extract lemmas from the processed tokens and print the first 50.

    # A better approach for this task is to pass the text directly to nlp (which tokenizes internally).
    # Then iterate through the doc's tokens and get their lemmas.
    current_book_lemmas = []
    for token in nlp(text.lower()): # Process the lowercased text directly
        # Filter out punctuation and spaces as we did for tokenization
        if token.is_alpha:
            current_book_lemmas.append(token.lemma_)

    lemmatized_books.append(current_book_lemmas)

    print(f"--- Book {i+1} (First 50 lemmatized tokens) ---")
    print(current_book_lemmas[:50])
    print("\n")

--- Book 1 (First 50 lemmatized tokens) ---
['chapter', 'down', 'the', 'rabbithole', 'chapter', 'ii', 'the', 'pool', 'of', 'tears', 'chapter', 'iii', 'a', 'caucusrace', 'and', 'a', 'long', 'tale', 'chapter', 'iv', 'the', 'rabbit', 'send', 'in', 'a', 'little', 'bill', 'chapter', 'advice', 'from', 'a', 'caterpillar', 'chapter', 'vi', 'pig', 'and', 'pepper', 'chapter', 'vii', 'a', 'mad', 'teaparty', 'chapter', 'viii', 'the', 'queen', 'croquetground', 'chapter', 'ix', 'the']


--- Book 2 (First 50 lemmatized tokens) ---
['chapter', 'lookingglass', 'house', 'chapter', 'ii', 'the', 'garden', 'of', 'live', 'flower', 'chapter', 'iii', 'lookingglass', 'insect', 'chapter', 'iv', 'tweedledum', 'and', 'tweedledee', 'chapter', 'wool', 'and', 'water', 'chapter', 'vi', 'humpty', 'dumpty', 'chapter', 'vii', 'the', 'lion', 'and', 'the', 'unicorn', 'chapter', 'viii', 'its', 'my', 'own', 'invention', 'chapter', 'ix', 'queen', 'alice', 'chapter', 'shake', 'chapter', 'xi', 'wake', 'chapter']


--- Book 3 (

### Task 7: Part-of-Speech (POS) Tagging

We'll use NLTK's averaged perceptron tagger to tag the first 50 tokens of each book with their part of speech.

In [ ]:
import nltk

# Download the POS tagger model if not already present
try:
    nltk.data.find('taggers/averaged_perceptron_tagger_eng')
except LookupError:
    nltk.download('averaged_perceptron_tagger_eng')

pos_tagged_books = []

for i, tokens in enumerate(tokenized_books):
    tagged = nltk.pos_tag(tokens)
    pos_tagged_books.append(tagged)

    print(f"--- Book {i+1} (First 50 POS-tagged tokens) ---")
    print(tagged[:50])
    print("\n")

### Task 8: Named Entity Recognition (NER)

We'll use spaCy to extract named entities (people, places, organizations, etc.) from each book.

In [ ]:
import spacy

# spaCy model 'nlp' was already loaded in Task 6; reload defensively in case cells are run out of order
try:
    nlp
except NameError:
    nlp = spacy.load("en_core_web_sm")

entities_books = []

for i, text in enumerate(books_corpus):
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    entities_books.append(entities)

    print(f"--- Book {i+1} (First 30 named entities) ---")
    print(entities[:30])
    print("\n")

### Task 9: Word Cloud Generation

We'll generate a word cloud for each book using the lemmatized, stopword-free tokens to visualize the most frequent words.

In [ ]:
import subprocess, sys
try:
    import wordcloud
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "wordcloud"])

from wordcloud import WordCloud
import matplotlib.pyplot as plt

book_titles = ["Alice's Adventures in Wonderland", "Through the Looking-Glass", "A Tangled Tale"]

for i, lemmas in enumerate(lemmatized_books):
    text_for_cloud = " ".join(lemmas)
    wc = WordCloud(width=800, height=400, background_color="white").generate(text_for_cloud)

    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Word Cloud - {book_titles[i]}")
    plt.show()

### Task 10: Bag of Words (BoW) Analysis

We'll build a Bag of Words representation with scikit-learn's `CountVectorizer` and inspect the most frequent terms in each book.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Rebuild documents from the lemmatized, stopword-free tokens so BoW works on clean units
bow_documents = [" ".join(lemmas) for lemmas in lemmatized_books]

count_vectorizer = CountVectorizer(stop_words="english", max_features=1000)
bow_matrix = count_vectorizer.fit_transform(bow_documents)

bow_df = pd.DataFrame(bow_matrix.toarray(), columns=count_vectorizer.get_feature_names_out(),
                       index=book_titles)

for i, title in enumerate(book_titles):
    top_words = bow_df.loc[title].sort_values(ascending=False).head(15)
    print(f"--- Book {i+1} ({title}) — Top 15 words by raw count ---")
    print(top_words)
    print("\n")

### Task 11: TF-IDF Analysis

We'll compute TF-IDF scores with `TfidfVectorizer` to find words that are distinctively important to each book, rather than just frequent across all three.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
tfidf_matrix = tfidf_vectorizer.fit_transform(bow_documents)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out(),
                         index=book_titles)

for i, title in enumerate(book_titles):
    top_tfidf = tfidf_df.loc[title].sort_values(ascending=False).head(15)
    print(f"--- Book {i+1} ({title}) — Top 15 words by TF-IDF score ---")
    print(top_tfidf)
    print("\n")

print("Note: BoW favors words that are merely frequent (often still generic nouns/verbs),\n"
      "while TF-IDF surfaces words that are distinctive to one book relative to the others\n"
      "(e.g. character names or book-specific vocabulary), which is the whole point of the metric.")